# Video analytics with Intel® Deep Learning Streamer (DL Streamer): Car detection and color classification

Step-by-step **DL Streamer** pipelines on a traffic clip: **car detection** and **color classification**; **OpenVINO™** runs the IR models.
Short cells below — pipeline text lives in `utils.py`. Close the preview window when a step ends (expected). `run_visual()` keeps console output quiet; FPS still appears on the video.


## Pipeline stages

```mermaid
flowchart LR
  VF[Video feed] --> Decode[Decode]
  Decode --> Pre[Pre-Process]
  Pre --> Inf[Inference]
  Inf --> Post[Post-Process]
  Post --> Enc[Encode]
  Enc --> AV[Annotated video]
```


## Configuration

- **Precision** → IR under `car-model/{FP16|FP32|INT8}/deployment/.../model.xml`.
- **Devices** → `gvadetect` / `gvaclassify` `device=` (dropdown uses OpenVINO when available).
- **Remote paths** (optional): before **Apply**, set `export DLSTREAMER_WORKDIR=/your/.../2` and `export DLSTREAMER_MODEL_ROOT=/your/.../car-model` (or edit defaults in `utils.py`).
- **Decoder**: default is `decodebin3` (same as *Video Analytics with DL-StreamerV1*). If your GStreamer has no `decodebin3`, run `export GST_DECODEBIN=decodebin` before Jupyter.

Run the cell below (opens dropdowns); **Apply** refreshes env. If `import utils` fails, set the kernel working directory to this folder.


In [ ]:
from utils import show_config_widgets

show_config_widgets()


### Verify `gvadetect`


In [ ]:
from utils import check_gvadetect
check_gvadetect()


## DEBUG — remove this section later

Temporary checks: paths, files, GStreamer / DL Streamer binaries, and a sample pipeline string. Delete this heading and the next cell when you no longer need it.

In [ ]:
# DEBUG — remove this cell later
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Ensure utils is importable (same folder as notebook)
for d in (Path.cwd(), Path.cwd() / "2"):
    if (d / "utils.py").is_file():
        if str(d) not in sys.path:
            sys.path.insert(0, str(d))
        break

import utils as u

u.apply_environment()

def _run(cmd: list[str], timeout: float = 15) -> tuple[int, str]:
    try:
        r = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        out = (r.stdout or "") + (r.stderr or "")
        return r.returncode, out.strip()[:4000]
    except Exception as e:
        return -1, str(e)

print("=== Python ===", sys.executable, sys.version.split()[0])
print("=== CWD ===", Path.cwd())

for name in ("gst-launch-1.0", "gst-inspect-1.0"):
    p = shutil.which(name)
    print(f"=== which {name} ===", p or "(not found)")

if shutil.which("gst-launch-1.0"):
    rc, out = _run(["gst-launch-1.0", "--version"])
    print("=== gst-launch-1.0 --version ===", f"rc={rc}\n", out[:800])

for el in ("decodebin3", "decodebin", "gvadetect", "gvaclassify", "gvatrack"):
    rc, out = _run(["gst-inspect-1.0", el])
    ok = rc == 0
    print(f"=== gst-inspect {el} ===", "OK" if ok else f"FAIL rc={rc}")
    if not ok and out:
        print(out[:500])

print("=== Env (DL Streamer / overrides) ===")
for k in sorted(os.environ):
    if k.startswith(("DLSTREAMER_", "GST_", "OPENVINO", "OV_")):
        print(f"  {k}={os.environ[k]!r}")

print("=== Key paths (from utils / env) ===")
keys = (
    "WORKDIR",
    "VIDEO_DIR",
    "VIDEO_SRC",
    "MODEL_DIR",
    "DETECTION_MODEL",
    "CLASSIFICATION_MODEL",
    "DETECTION_DEVICE",
    "CLASSIFICATION_DEVICE",
    "PRECISION",
    "GST_DECODEBIN",
)
for k in keys:
    v = os.environ.get(k, "(unset)")
    exists = ""
    if k.endswith("MODEL") or k.endswith("SRC") or k in ("VIDEO_DIR",):
        p = Path(v) if v != "(unset)" else None
        if p is not None:
            exists = f"  [{'FILE' if p.is_file() else 'DIR' if p.is_dir() else 'MISSING'}]"
    print(f"  {k}={v}{exists}")

print("=== Sample pipeline (first step) ===")
print(u.pipeline_raw_video())


### 1. Decode + display


In [ ]:
from utils import run_visual, pipeline_raw_video
run_visual(pipeline_raw_video())


### 2. + FPS overlay (`gvafpscounter` → `autovideosink`)


In [ ]:
from utils import run_visual, pipeline_fps
run_visual(pipeline_fps())


### 3. Vehicle detection


In [ ]:
from utils import run_visual, pipeline_detection
run_visual(pipeline_detection())


### 4. Detection + color classification


In [ ]:
from utils import run_visual, pipeline_detect_classify
run_visual(pipeline_detect_classify())


### 5. Benchmark: 2 streams (`fakesink`)


In [ ]:
from utils import run_benchmark, pipeline_benchmark_2
run_benchmark(pipeline_benchmark_2())


### 6. Benchmark: 4 streams


In [ ]:
from utils import run_benchmark, pipeline_benchmark_4
run_benchmark(pipeline_benchmark_4())
